# Milestone 4 — Zhuang MERFISH cross-reference

Mean log2(CPM+1) per **cell type** × **brain area** on Zhuang whole-brain MERFISH replicates,
then correlation scatter vs Allen MERFISH (notebook 02). Driven by `receptor_query_config.yaml`.

Zhuang replicates are averaged (mean per cell_type × region × gene). Allen MERFISH loads from a
prior `aggregated_merfish.parquet` in `exploration/` when available; otherwise re-runs with a warning.
Allen imputed genes are marked with `*` in combined heatmaps and hollow markers in cross-ref scatters.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import (
    DEFAULT_OUTPUT_DIR,
    get_zhuang_datasets,
    load_config,
    restrict_config_to_genes,
    start_run,
)
from src.data_loaders import (
    get_abc_cache,
    load_zhuang_cell_metadata,
    check_zhuang_genes,
    load_zhuang_aggregated,
    load_allen_merfish_aggregate,
    merfish_gene_source_map,
    merge_crossref_aggregates,
    family_gene_region_matrix_merfish,
    combined_heatmap_matrix,
)
from src.plotting import (
    plot_family_heatmap,
    plot_combined_heatmap,
    plot_crossref_family_scatters,
    IMPUTED_GENE_MARKER,
)

In [ ]:
EXPLORATION_ROOT = DEFAULT_OUTPUT_DIR

CONFIG_PATH = PROJECT_ROOT / "receptor_query_config.yaml"
config = load_config(CONFIG_PATH)
config["_dataset_modality"] = "zhuang"

zhuang_datasets = get_zhuang_datasets(config)
config["_zhuang_replicates_used"] = zhuang_datasets

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="Zhuang-ABCA",
    exploration_root=EXPLORATION_ROOT,
    notebook="04_zhuang_crossref",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)

print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
filt = config.get("cell_type_name_filter") or []
print(f"Cell type name filter: {filt if filt else '(none — all types)'}")
print(f"Zhuang replicates: {zhuang_datasets}")
print(f"Run dir: {OUTPUT_DIR}")
print(f"Manifest: {OUTPUT_DIR / 'run_manifest.json'}")

In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

In [ ]:
for dataset_id in zhuang_datasets:
    cell_meta = load_zhuang_cell_metadata(cache, config, dataset_id)
    print(f"\n{dataset_id}: {len(cell_meta):,} cells in config brain areas")
    print(cell_meta.groupby("brain_area").size().sort_values(ascending=False))

In [ ]:
requested = list(config["_all_genes"])
genes_flat_orig = dict(config["_genes_flat"])
panel_ref = zhuang_datasets[0]
availability = check_zhuang_genes(cache, requested, panel_ref)

print(f"In Zhuang panel ({panel_ref}, n≈1122): {len(availability['present'])}")
print(f"Missing ({len(availability['missing'])}):")
for gene in availability["missing"]:
    print(f"  {gene} ({genes_flat_orig.get(gene, 'unknown')})")

if not availability["present"]:
    raise RuntimeError("No requested genes in Zhuang MERFISH panel.")

agg_long = load_zhuang_aggregated(cache, config)
loaded_genes = sorted(agg_long["gene"].unique())
restrict_config_to_genes(config, loaded_genes)

print(f"\nProceeding with {len(config['_all_genes'])} / {len(requested)} genes.")
print(f"Families with data: {config['_families']}")
print(f"Aggregated rows (mean across replicates): {len(agg_long):,}")
print(agg_long.head())

In [ ]:
print(f"Saving Zhuang heatmaps to {OUTPUT_DIR}")

for family in config["_families"]:
    mat = family_gene_region_matrix_merfish(agg_long, family, config)
    if mat.empty:
        warnings.warn(f"No data for family {family!r}; skipping heatmap.")
        continue
    path = plot_family_heatmap(family, mat, config, output_dir=OUTPUT_DIR)
    print(f"Saved {path}")

In [ ]:
combined = combined_heatmap_matrix(agg_long, config)
print(f"Combined heatmap: {combined.shape[0]} cell types × {combined.shape[1]} genes")
path = plot_combined_heatmap(combined, config, output_dir=OUTPUT_DIR)
print(f"Saved {path}")

In [ ]:
if config["output"].get("save_processed_data", True):
    parquet_path = OUTPUT_DIR / "aggregated_zhuang.parquet"
    agg_long.to_parquet(parquet_path, index=False)
    print(f"Saved aggregated Zhuang matrix to {parquet_path}")

## Cross-reference: Allen MERFISH vs Zhuang

Loads Allen MERFISH aggregates from the newest matching run folder under `exploration/`
(pattern: `{timestamp}_{cell_type_level}_{merfish_dataset}/aggregated_merfish.parquet`).
Falls back to re-running Allen aggregation with a marked warning if not found.

In [ ]:
allen_agg, allen_parquet_path, allen_reran = load_allen_merfish_aggregate(
    cache,
    config,
    exploration_root=EXPLORATION_ROOT,
)

if allen_parquet_path is not None:
    print(f"Loaded Allen MERFISH aggregates from:\n  {allen_parquet_path}")
else:
    print("Allen MERFISH aggregates computed in this session (no prior parquet found).")

overlap_genes = sorted(set(allen_agg["gene"]) & set(agg_long["gene"]))
print(f"\nOverlapping genes for cross-ref: {len(overlap_genes)} / {len(config['_all_genes'])}")

allen_sources = merfish_gene_source_map(cache, overlap_genes, config)
config["_allen_gene_sources"] = allen_sources
n_imputed = sum(1 for s in allen_sources.values() if s == "imputed")
if n_imputed:
    imputed_genes = [g for g, s in allen_sources.items() if s == "imputed"]
    print(f"Allen imputed genes in overlap ({n_imputed}): "
          f"{[g + IMPUTED_GENE_MARKER for g in imputed_genes[:12]]}"
          f"{'...' if n_imputed > 12 else ''}")

crossref = merge_crossref_aggregates(allen_agg, agg_long, allen_sources)
print(f"Cross-reference rows (cell_type × brain_area × gene): {len(crossref):,}")
if crossref.empty:
    raise RuntimeError("No overlapping Allen/Zhuang expression rows for cross-reference.")
print(crossref.head())

In [ ]:
paths = plot_crossref_family_scatters(crossref, config, output_dir=OUTPUT_DIR)
for path in paths:
    print(f"Saved {path}")

In [ ]:
if config["output"].get("save_processed_data", True):
    crossref_path = OUTPUT_DIR / "crossref_allen_zhuang.parquet"
    crossref.to_parquet(crossref_path, index=False)
    print(f"Saved cross-reference table to {crossref_path}")

---
## Dev / smoke test (2 genes × 2 regions × 1 replicate)

Run this cell only to validate the pipeline with a smaller download footprint.

In [ ]:
# test_config = load_config(CONFIG_PATH)
# test_config["brain_areas"] = ["STR", "TH"]
# test_config["receptors"] = {"dopamine": ["Drd1", "Drd2"]}
# genes_map = {}
# for fam, glist in test_config["receptors"].items():
#     for g in glist:
#         genes_map[g] = fam
# test_config["_genes_flat"] = genes_map
# test_config["_all_genes"] = list(genes_map)
# test_config["_families"] = list(test_config["receptors"].keys())
# test_config["data"]["zhuang_datasets"] = ["Zhuang-ABCA-4"]
# test_config["_dataset_modality"] = "zhuang"
# test_config["_zhuang_replicates_used"] = test_config["data"]["zhuang_datasets"]
#
# test_cache = get_abc_cache(test_config)
# test_agg = load_zhuang_aggregated(test_cache, test_config)
# test_mat = family_gene_region_matrix_merfish(test_agg, "dopamine", test_config)
# plot_family_heatmap("dopamine", test_mat, test_config, output_dir=OUTPUT_DIR)
# print(f"Test heatmap saved to {OUTPUT_DIR / 'heatmap_dopamine.png'}")